!pip3 install drain3 pandas regex tqdm

# Data Exploration

In [1]:
!pip3 install drain3 pandas regex



[notice] A new release of pip is available: 23.1.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.1.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
# Install Drain3

import os
import pandas as pd
import regex as re
import urllib.request
import ssl
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

# Download HDFS_2k.log sample from Loghub
data_url = "https://raw.githubusercontent.com/logpai/loghub/master/HDFS/HDFS_2k.log"
log_file = "HDFS_2k.log"

if not os.path.exists(log_file):
    print(f"Downloading {log_file}...")
    # Create unverified SSL context to bypass certificate errors
    ssl_context = ssl._create_unverified_context()
    with urllib.request.urlopen(data_url, context=ssl_context) as response, open(log_file, 'wb') as out_file:
        out_file.write(response.read())
    print("Download complete.")
else:
    print(f"{log_file} already exists.")

# Display first few lines
with open(log_file, 'r') as f:
    head = [next(f) for _ in range(5)]
print("Sample logs:")
print("".join(head))

HDFS_2k.log already exists.
Sample logs:
081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating
081109 203807 222 INFO dfs.DataNode$PacketResponder: PacketResponder 0 for block blk_-6952295868487656571 terminating
081109 204005 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.73.220:50010 is added to blk_7128370237687728475 size 67108864
081109 204015 308 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_8229193803249955061 terminating
081109 204106 329 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_-6670958622368987959 terminating



In [3]:
# Download all Loghub datasets
import os
import urllib.request
import ssl

# Create Data directory if it doesn't exist
data_dir = "Data2k"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# List of datasets from Loghub (2k samples)
datasets = {
    "HDFS": "https://raw.githubusercontent.com/logpai/loghub/master/HDFS/HDFS_2k.log",
    "Hadoop": "https://raw.githubusercontent.com/logpai/loghub/master/Hadoop/Hadoop_2k.log",
    "Spark": "https://raw.githubusercontent.com/logpai/loghub/master/Spark/Spark_2k.log",
    "Zookeeper": "https://raw.githubusercontent.com/logpai/loghub/master/Zookeeper/Zookeeper_2k.log",
    "BGL": "https://raw.githubusercontent.com/logpai/loghub/master/BGL/BGL_2k.log",
    "HPC": "https://raw.githubusercontent.com/logpai/loghub/master/HPC/HPC_2k.log",
    "Thunderbird": "https://raw.githubusercontent.com/logpai/loghub/master/Thunderbird/Thunderbird_2k.log",
    "Windows": "https://raw.githubusercontent.com/logpai/loghub/master/Windows/Windows_2k.log",
    "Linux": "https://raw.githubusercontent.com/logpai/loghub/master/Linux/Linux_2k.log",
    "Android": "https://raw.githubusercontent.com/logpai/loghub/master/Android/Android_2k.log",
    "HealthApp": "https://raw.githubusercontent.com/logpai/loghub/master/HealthApp/HealthApp_2k.log",
    "Apache": "https://raw.githubusercontent.com/logpai/loghub/master/Apache/Apache_2k.log",
    "Proxifier": "https://raw.githubusercontent.com/logpai/loghub/master/Proxifier/Proxifier_2k.log",
    "OpenSSH": "https://raw.githubusercontent.com/logpai/loghub/master/OpenSSH/OpenSSH_2k.log",
    "OpenStack": "https://raw.githubusercontent.com/logpai/loghub/master/OpenStack/OpenStack_2k.log",
    "Mac": "https://raw.githubusercontent.com/logpai/loghub/master/Mac/Mac_2k.log"
}

# Create unverified SSL context
ssl_context = ssl._create_unverified_context()

print(f"Downloading datasets to {data_dir}/...")

for name, url in datasets.items():
    file_path = os.path.join(data_dir, f"{name}_2k.log")
    if not os.path.exists(file_path):
        try:
            print(f"Downloading {name}...")
            with urllib.request.urlopen(url, context=ssl_context) as response, open(file_path, 'wb') as out_file:
                out_file.write(response.read())
        except Exception as e:
            print(f"Failed to download {name}: {e}")
    else:
        print(f"{name} already exists.")

print("All downloads complete.")

HDFS already exists.
Hadoop already exists.
Spark already exists.
Zookeeper already exists.
BGL already exists.
HPC already exists.
Thunderbird already exists.
Windows already exists.
Linux already exists.
Android already exists.
HealthApp already exists.
Apache already exists.
Proxifier already exists.
OpenSSH already exists.
OpenStack already exists.
Mac already exists.
All downloads complete.


In [4]:
# Configure Drain3
config = TemplateMinerConfig()
# config.load_default_config() # Removed: Not needed/available in this version
config.profiling_enabled = False
template_miner = TemplateMiner(config=config)

# Regex to extract BlockId (HDFS specific)
block_id_pattern = re.compile(r'(blk_[-0-9]+)')

parsed_data = []

print("Parsing logs...")
with open(log_file, 'r') as f:
    for line_count, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
            
        # 1. Extract BlockId
        match = block_id_pattern.search(line)
        block_id = match.group(1) if match else "Unknown"
        
        # 2. Parse with Drain3
        # We process the message part. For HDFS, the content usually starts after some timestamp/level info.
        # However, Drain3 can handle full lines, but it's better to strip headers if possible.
        # For simplicity, we'll pass the full line or a simple split.
        # HDFS format: <Date> <Time> <Pid> <Level> <Component>: <Content>
        # Example: 081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating
        
        # Simple heuristic: take everything after the first colon if present, else full line
        content = line.split(':', 1)[1].strip() if ':' in line else line
        
        result = template_miner.add_log_message(content)
        
        parsed_data.append({
            'LineId': line_count + 1,
            'BlockId': block_id,
            'RawContent': content,
            'EventId': result['cluster_id'],
            'EventTemplate': result['template_mined']
        })

print(f"Finished parsing {len(parsed_data)} lines.")
print(f"Found {len(template_miner.drain.clusters)} unique templates.")

# Convert to DataFrame
df_parsed = pd.DataFrame(parsed_data)
df_parsed.head()

Parsing logs...
Finished parsing 2000 lines.
Found 17 unique templates.


,LineId,BlockId,RawContent,EventId,EventTemplate
0,1,blk_38865049064139660,PacketResponder 1 for block blk_38865049064139...,1,PacketResponder 1 for block blk_38865049064139...
1,2,blk_-6952295868487656571,PacketResponder 0 for block blk_-6952295868487...,1,PacketResponder <*> for block <*> terminating
2,3,blk_7128370237687728475,BLOCK* NameSystem.addStoredBlock: blockMap upd...,2,BLOCK* NameSystem.addStoredBlock: blockMap upd...
3,4,blk_8229193803249955061,PacketResponder 2 for block blk_82291938032499...,1,PacketResponder <*> for block <*> terminating
4,5,blk_-6670958622368987959,PacketResponder 2 for block blk_-6670958622368...,1,PacketResponder <*> for block <*> terminating


In [5]:
# Display extracted templates
print("Extracted Templates:")
templates_df = df_parsed[['EventId', 'EventTemplate']].drop_duplicates().sort_values('EventId')
for _, row in templates_df.iterrows():
    print(f"ID {row['EventId']}: {row['EventTemplate']}")

# Create Log Sequences
# Group by BlockId and collect EventIds
sequence_df = df_parsed.groupby('BlockId')['EventId'].apply(list).reset_index()
sequence_df.rename(columns={'EventId': 'EventSequence'}, inplace=True)

# Calculate sequence length
sequence_df['Length'] = sequence_df['EventSequence'].apply(len)

print("\nGenerated Sequences:")
print(sequence_df.head())

# Example of a full sequence
sample_block = sequence_df.iloc[0]
print(f"\nBlock: {sample_block['BlockId']}")
print(f"Sequence: {sample_block['EventSequence']}")
print("Reconstructed Sequence (Templates):")
for eid in sample_block['EventSequence']:
    template = templates_df[templates_df['EventId'] == eid]['EventTemplate'].values[0]
    print(f"  -> {template}")

Extracted Templates:
ID 1: PacketResponder 1 for block blk_38865049064139660 terminating
ID 1: PacketResponder <*> for block <*> terminating
ID 2: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.73.220:50010 is added to blk_7128370237687728475 size 67108864
ID 2: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size 67108864
ID 2: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>
ID 3: Received block blk_3587508140051953248 of size 67108864 from /10.251.42.84
ID 3: Received block <*> of size 67108864 from <*>
ID 3: Received block <*> of size <*> from <*>
ID 4: Receiving block blk_5792489080791696128 src: /10.251.30.6:33145 dest: /10.251.30.6:50010
ID 4: Receiving block <*> src: <*> dest: <*>
ID 5: BLOCK* NameSystem.allocateBlock: /user/root/rand/_temporary/_task_200811092030_0001_m_000590_0/part-00590. blk_-1727475099218615100
ID 5: BLOCK* NameSystem.allocateBlock: <*> <*>
ID 6: Verification succeeded for blk_-49809165198

## Recommendation
Start with Option A (Separate Parsers) using Drain3. It is the industry standard for a reason (fast, accurate, online).

Create a configuration file/object for each log source.
Define custom regex masking for each source (e.g., HDFS has BlockIDs, Apache has UserAgents).
Parse them independently to get clean EventSequences.
Merge the resulting sequences only at the feature vector level if you want a unified model, or keep them separate for source-specific models.


## TODO

- create custom parser class that will take the configuration from file for specific logs type/source
- consider working only on the most stripped down template for embeddings
- condider creation of sequences and final template format

3. Handling Labeled vs. Unlabeled Data
The parsing stage is unsupervised, so it doesn't care if data is labeled or not.

Parsing: Run all data (labeled and unlabeled) through the parser to build a comprehensive set of templates.
Training (Representation Learning): Use the sequences from all data (labeled + unlabeled) to train your embeddings (e.g., Word2Vec, Log2Vec, BERT). Unsupervised learning benefits from more data.
Training (Anomaly Detection):
Supervised: Use only the labeled sequences to train a classifier (e.g., LSTM classifier).
Unsupervised/Semi-supervised: Use normal data (often assumed to be the majority of unlabeled data) to train a reconstruction model (e.g., Autoencoder, DeepLog).

In [7]:
!pip3 install tqdm


[notice] A new release of pip is available: 23.1.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.1.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


# BGL dataset processing

In [21]:
from Parser.log_parser import LogParser
import os
import pandas as pd
from tqdm import tqdm
import importlib
import Parser.log_parser

# Reload the module to ensure latest changes (fixes) are picked up
importlib.reload(Parser.log_parser)
from Parser.log_parser import LogParser

# 1. Select dataset to parse
# Options: HDFS, Hadoop, Spark, Zookeeper, BGL, HPC, Thunderbird, Windows, Linux, Android, HealthApp, Apache, Proxifier, OpenSSH, OpenStack, Mac
dataset_name = "BGL" 
log_file_path = f"Data/{dataset_name}.log"

# 2. Initialize the parser wrapper
parser = LogParser(dataset_name)

parsed_logs = []

# 3. Process the file
if os.path.exists(log_file_path):
    print(f"Reading {log_file_path}...")
    with open(log_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    
    print(f"Parsing {len(lines)} lines...")
    
    # Iterate over lines with progress bar
    for idx, line in enumerate(tqdm(lines)):
        line = line.strip()
        if not line:
            continue
            
        # Extract Label (BGL specific: first column)
        # Format: Label Timestamp ...
        # If the first token is "-", it's normal. Otherwise it's an alert label.
        parts = line.split(' ', 1)
        label = parts[0]
        content = parts[1] if len(parts) > 1 else line
        
        # Parse the content (excluding the label)
        result = parser.parse(content)
        
        parsed_logs.append({
            'LineId': idx + 1,
            'Label': label != '-',
            'EventId': result['cluster_id'],
            'EventTemplate': result['template_mined'], # This is the template at this specific moment
            'RawLog': content
        })
        
    print("Parsing complete.")
    
    # Convert to DataFrame
    df_parsed = pd.DataFrame(parsed_logs)
    
    # Update to use the FINAL (most general) template for each EventId
    # This ensures consistency across the dataset
    all_clusters = parser.get_templates()
    cluster_map = {c.cluster_id: c.get_template() for c in all_clusters}
    df_parsed['EventTemplate'] = df_parsed['EventId'].map(cluster_map)
    
    print(f"Total templates identified: {len(all_clusters)}")
    
else:
    print(f"File {log_file_path} not found. Please run the download cell first.")

df_parsed.head()

Loading configuration from /Users/michalklos/Documents/Studia2025/Magisterka/Notebooks/Parser/BGL.json
Reading Data/BGL.log...
Parsing 4747963 lines...
Parsing 4747963 lines...


100%|██████████| 4747963/4747963 [05:50<00:00, 13532.99it/s]



Parsing complete.
Total templates identified: 1347
Total templates identified: 1347


,LineId,Label,EventId,EventTemplate,RawLog
0,1,False,1,<*> <*> <*> <*> <*> RAS KERNEL INFO instructio...,1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005...
1,2,False,1,<*> <*> <*> <*> <*> RAS KERNEL INFO instructio...,1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005...
2,3,False,1,<*> <*> <*> <*> <*> RAS KERNEL INFO instructio...,1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005...
3,4,False,1,<*> <*> <*> <*> <*> RAS KERNEL INFO instructio...,1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005...
4,5,False,1,<*> <*> <*> <*> <*> RAS KERNEL INFO instructio...,1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005...


In [22]:
# Analyze the extracted templates
if not df_parsed.empty:
    # Get unique templates
    templates_df = df_parsed[['EventId', 'EventTemplate']].drop_duplicates().sort_values('EventId')
    
    print(f"--- Top 10 Templates for {dataset_name} ---")
    for index, row in templates_df.head(10).iterrows():
        print(f"ID {row['EventId']}: {row['EventTemplate']}")
        
    # Show distribution of events
    print("\n--- Top 5 Most Frequent Events ---")
    print(df_parsed['EventId'].value_counts().head(5))
    
    # Show Label distribution
    print("\n--- Label Distribution ---")
    print(df_parsed['Label'].value_counts())
    
    # Example of full reconstruction
    print("\n--- Sample Reconstruction ---")
    sample_row = df_parsed.iloc[0]
    print(f"Raw:      {sample_row['RawLog']}")
    print(f"Label:    {sample_row['Label']}")
    print(f"Template: {sample_row['EventTemplate']}")

--- Top 10 Templates for BGL ---
ID 1: <*> <*> <*> <*> <*> RAS KERNEL INFO instruction cache parity error corrected
ID 2: <*> <*> <*> <*> <*> RAS LINKCARD INFO MidplaneSwitchController performing bit sparing on <*> bit <*>
ID 3: <*> <*> <*> <*> <*> RAS KERNEL INFO generating <<CORE>>
ID 4: <*> <*> <*> <*> <*> RAS KERNEL INFO <*> ddr errors(s) detected and corrected on rank 0, symbol <*> bit <*>
ID 5: <*> <*> <*> <*> <*> RAS KERNEL INFO <*> <*> <*> error(s) (dcr <<HEX>>) detected and corrected
ID 6: <*> <*> <*> <*> <*> RAS KERNEL INFO CE sym <*> at <<HEX>>, mask <<HEX>>
ID 7: <*> <*> <*> <*> <*> RAS KERNEL INFO total of <*> ddr error(s) detected and corrected
ID 8: <*> <*> <*> <*> <*> RAS KERNEL INFO ddr: activating redundant bit steering: rank=0 <*>
ID 9: <*> <*> <*> <*> <*> RAS KERNEL INFO ddr: excessive soft failures, consider replacing the card
ID 10: <*> <*> <*> <*> <*> RAS APP FATAL ciod: Error loading <*> invalid or missing program image, No such file or directory

--- Top 5 Most

In [19]:
!pip3 install fastparquet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 909.9/909.9 kB 6.9 MB/s eta 0:00:0000:0100:01     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/909.9 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 909.9/909.9 kB 6.9 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.9 MB ? eta -:--:--  Downloading cramjam-2.11.0-cp311-cp311-macosx_10_12_x86_64.whl (1.9 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.4 MB/s eta 0:00:0000:01

[notice] A new release of pip is available: 23.1.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.1.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Save parsed logs data to parquet data format

In [23]:
import os

# Ensure output directory exists
output_dir = "Output"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_parquet = f"{output_dir}/{dataset_name}_parsed.parquet"

try:
    # Try using fastparquet engine to avoid pyarrow conflict
    df_parsed.to_parquet(output_parquet, index=False, engine='fastparquet')
    print(f"Saved parsed data to {output_parquet} (using fastparquet)")
except Exception as e:
    print(f"Fastparquet failed: {e}")

Saved parsed data to Output/BGL_parsed.parquet (using fastparquet)


## TODO
**-Parse to more distinct features**
